# <b>Final Project - Job Application Helper (Resume Screening / Recommending Job / Comparing Job Description and Resume)</b>

#### Importing the necessary libraries

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#### Importing the dataset into dataframe and displaying the first 5 rows in it.

In [13]:
df1 = pd.read_csv('ResumeDataSet.csv')
df1.head()

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


In [14]:
# Rename the column
df1.rename(columns={'Label': 'Category'}, inplace=True)
df1.head()


,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


# <b>Data Exploration</b>

- Checking if the dataset has any null values or not

In [15]:
df1.isnull().sum()


Category    0
Resume      0
dtype: int64

In [16]:
df1.dropna(inplace=True)
df1.isnull().sum()

Category    0
Resume      0
dtype: int64

- Finding basic information about the dataset
    - It has 29033 rows and 2 columns
    - Both the columns contains data of Object datatype i.e. string

In [17]:
print(df1.shape)
print(df1.info())

(962, 2)
<class 'pandas.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  962 non-null    str  
 1   Resume    962 non-null    str  
dtypes: str(2)
memory usage: 15.2 KB
None


- Looking at the Unique Categories

In [18]:
unique_categories = df1['Category'].unique()
print(unique_categories)

<StringArray>
[             'Data Science',                        'HR',
                  'Advocate',                      'Arts',
             'Web Designing',       'Mechanical Engineer',
                     'Sales',        'Health and fitness',
            'Civil Engineer',            'Java Developer',
          'Business Analyst',             'SAP Developer',
        'Automation Testing',    'Electrical Engineering',
        'Operations Manager',          'Python Developer',
           'DevOps Engineer', 'Network Security Engineer',
                       'PMO',                  'Database',
                    'Hadoop',             'ETL Developer',
          'DotNet Developer',                'Blockchain',
                   'Testing']
Length: 25, dtype: str


- Exploring Categories - getting value counts of each category. 

In [19]:
df1['Category'].value_counts()

Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
Data Science                 40
Mechanical Engineer          40
Sales                        40
Operations Manager           40
ETL Developer                40
Blockchain                   40
Arts                         36
Database                     33
Health and fitness           30
Electrical Engineering       30
PMO                          30
Business Analyst             28
DotNet Developer             28
Automation Testing           26
Network Security Engineer    25
Civil Engineer               24
SAP Developer                24
Advocate                     20
Name: count, dtype: int64

In [20]:
temp_df = df1

# <b>Data Processing</b> 

In [21]:
# Preprocessing libraries
import re
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer 
import string
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asrit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\asrit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Cleaning Data:                                     
1 Removing (URLs, hashtags, mentions, special letters, punctuations)

2 Tokenizing the cleaned text

3 Removing Stop Words

4 Performing Lemmatization on final text    

- The resumeKeywords function removes URLs, hashtags, mentions, special characters, non-ASCII characters, multiple spaces, and stop words from the input text while also performing tokenization, lowercasing, and lemmatization to provide cleaned and processed text as output.

- <b>Tokenization:</b> Tokenization is the process of breaking a text into individual words or tokens. In this step, the text is split into its constituent words, which makes it easier to analyze and process. For example, the sentence "I love coding" would be tokenized into three tokens: "I," "love," and "coding."

- <b> StopWords: </b> Stopwords are common words that are typically removed from text during natural language processing to improve text analysis and reduce noise in the data.Examples of common stopwords in English include "the," "and," "in," "is," "of," "it," "to," and many others. Removing stopwords from text helps reduce the dimensionality of the data and focuses the analysis on more meaningful words

- <b>Lemmatization:</b> Lemmatization is the process of reducing words to their base or root form. This step is essential for text analysis because it reduces different forms of a word to a common base form. For example, the words "running" and "ran" would both be lemmatized to "run." This simplifies the text and ensures that similar words are treated as the same, which is crucial for accurate analysis and modeling.

In [22]:

def resumeKeywords(txt):   # sourcery skip: avoid-builtin-shadow, list-comprehension
    cleanText = re.sub('http\S+\s', ' ', txt) # Removing URLs
    cleanText = re.sub('#\S+\s', ' ', cleanText) # Removing hashtags
    cleanText = re.sub('@\S+', '  ', cleanText)  # Removing mentions
    cleanText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', cleanText) # Removing punctuations
    cleanText = re.sub(r'[^\x00-\x7f]', ' ', cleanText) # Removing non-ASCII characters
    cleanText = re.sub('\s+', ' ', cleanText) # Replace multiple spaces with a single space
    cleanText = cleanText.strip() # Removing leading and trailing whitespaces
    
    #------------Tokenizing Cleaned Text--------------------------------------------------------
    # Tokenizing our cleaned text
    tokenizer = nltk.tokenize.RegexpTokenizer('\w+')
    tokens = tokenizer.tokenize(cleanText)
    # Now lower everything and storing it in new variable words
    words = []
    for word in tokens:
        words.append(word.lower())
    #--------------------------------------------------------------------------------------------

    #-------------Removing Stop Words------------------------------------------------------------
    stopwords = nltk.corpus.stopwords.words('english')
    words_new = []
    for word in words:
        if word not in stopwords:
            words_new.append(word)
    #--------------------------------------------------------------------------------------------
    #-----------Performing Lemmatization---------------------------------------------------------
    wn = WordNetLemmatizer() 
    lemm_text = [wn.lemmatize(word) for word in words_new]
    #--------------------------------------------------------------------------------------------
    #----------Converting List into String-------------------------------------------------------
    processed_text = ' '.join(lemm_text)
    
    return processed_text

- Testing the above custom function to remove certain details from Resume

In [23]:
resumeKeywords(" https://www.github.com Software Engineer with 2 years of experience in Data Structures and Algorithms Agile Scrum, SDLC, C++, Java MVC, JavaScript, Web Development, Python " +
               "Data Science, Machine Learning / AI, and Mainframe technologies Programming Languages: 	C/C++, Java, Python, SQL, JCL, Cobol, DB2" + 
               "Frameworks: %#####	Java Spring, Spring boot, React, Angular, NodeJs" +
               "Tools: 	GIT, Visual Studio Code, Sublime, Spyder, Jupyter Notebook, Bluezone, Netbeans, Jira, Confluence, Kanban, CI/CD (Jenkins, GitLab, Azure-Devops), AWS," + 
               "Data-Bricks Libraries: 	NumPy, Pandas, Matplotlib, nltk, Scikit learn, TensorFlow, Keras Other: 	Problem-Solving, Quick Learner, Time-Management")

'software engineer 2 year experience data structure algorithm agile scrum sdlc c java mvc javascript web development python data science machine learning ai mainframe technology programming language c c java python sql jcl cobol db2frameworks java spring spring boot react angular nodejstools git visual studio code sublime spyder jupyter notebook bluezone netbeans jira confluence kanban ci cd jenkins gitlab azure devops aws data brick library numpy panda matplotlib nltk scikit learn tensorflow kera problem solving quick learner time management'

#### Applying above created Custom Function to process the data and creating new column "Processed_Resume"

In [24]:
df1['Processed_Resume'] = df1['Resume'].apply(lambda x: resumeKeywords(x))
df1.head()

,Category,Resume,Processed_Resume
0,Data Science,Skills * Programming Languages: Python (pandas...,skill programming language python panda numpy ...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...,education detail may 2013 may 2017 b e uit rgp...
2,Data Science,"Areas of Interest Deep Learning, Control Syste...",area interest deep learning control system des...
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...,skill r python sap hana tableau sap hana sql s...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab...",education detail mca ymcaust faridabad haryana...


#### Generating WordCloud from the cleaned text

In [25]:
!pip install wordcloud
from wordcloud import WordCloud

  Using cached wordcloud-1.9.6-cp311-cp311-win_amd64.whl.metadata (3.5 kB)
Using cached wordcloud-1.9.6-cp311-cp311-win_amd64.whl (306 kB)


In [26]:
import plotly.express as px
import plotly.graph_objects as go
# Join the cleaned text into a single string
text = ' '.join(df1['Processed_Resume'])

# Create a word cloud
wordcloud = WordCloud(background_color='white',
                      width=1000,
                      height=800,
                      max_words=500,
                      colormap='viridis'
                      ).generate(text)

# Convert word cloud to an image
wordcloud_image = wordcloud.to_image()

# Display the word cloud using Plotly as an image
fig = px.imshow(wordcloud_image)
fig.update_layout(
    title='Word Cloud of Cleaned Text',
    xaxis_showticklabels=False,
    yaxis_showticklabels=False,
    plot_bgcolor='white'
)
import plotly.io as pio
pio.renderers.default = "browser"

fig.show()

#### Encoding the Category column and plotting it

In [27]:
# Label encoding our Category
label = LabelEncoder()
df1['Encoded_Category'] = label.fit_transform(df1['Category'])
df1.head()

,Category,Resume,Processed_Resume,Encoded_Category
0,Data Science,Skills * Programming Languages: Python (pandas...,skill programming language python panda numpy ...,6
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...,education detail may 2013 may 2017 b e uit rgp...,6
2,Data Science,"Areas of Interest Deep Learning, Control Syste...",area interest deep learning control system des...,6
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...,skill r python sap hana tableau sap hana sql s...,6
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab...",education detail mca ymcaust faridabad haryana...,6


In [28]:
df1['Category'].sample(15)

111                  Arts
289    Health and fitness
401        Java Developer
471    Automation Testing
73                     HR
766                Hadoop
333        Java Developer
819         ETL Developer
795         ETL Developer
556      Python Developer
430      Business Analyst
616       DevOps Engineer
942               Testing
817         ETL Developer
584      Python Developer
Name: Category, dtype: str

In [29]:
# Create the mapping
category_mapping = df1.groupby('Encoded_Category')['Category'].first().to_dict()

# Print the created mapping
print("Created Category Mapping:")
print(category_mapping)

Created Category Mapping:
{0: 'Advocate', 1: 'Arts', 2: 'Automation Testing', 3: 'Blockchain', 4: 'Business Analyst', 5: 'Civil Engineer', 6: 'Data Science', 7: 'Database', 8: 'DevOps Engineer', 9: 'DotNet Developer', 10: 'ETL Developer', 11: 'Electrical Engineering', 12: 'HR', 13: 'Hadoop', 14: 'Health and fitness', 15: 'Java Developer', 16: 'Mechanical Engineer', 17: 'Network Security Engineer', 18: 'Operations Manager', 19: 'PMO', 20: 'Python Developer', 21: 'SAP Developer', 22: 'Sales', 23: 'Testing', 24: 'Web Designing'}


### <b>#Vectorization (using TfidfVectorizer)</b>

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english',max_features=29033)

tfidf.fit(df1['Processed_Resume'])
requiredText  = tfidf.transform(df1['Processed_Resume'])
print(requiredText)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 155341 stored elements and shape (962, 6562)>
  Coords	Values
  (0, 118)	0.06453783899846323
  (0, 268)	0.04632697838273315
  (0, 298)	0.031126979945512483
  (0, 317)	0.025361912989866155
  (0, 340)	0.033319272390464866
  (0, 370)	0.029288428444891598
  (0, 495)	0.13903194496767213
  (0, 497)	0.07851286854091813
  (0, 500)	0.15656008367123686
  (0, 505)	0.02096423266463051
  (0, 512)	0.026190077390818307
  (0, 523)	0.04632697838273315
  (0, 526)	0.03727134831379874
  (0, 622)	0.04632697838273315
  (0, 640)	0.053810183291968954
  (0, 643)	0.08098518487138881
  (0, 645)	0.02899820504264113
  (0, 646)	0.06518168127424176
  (0, 650)	0.10712314267369262
  (0, 667)	0.03771995525102985
  (0, 700)	0.027461726023211917
  (0, 703)	0.03503454597093106
  (0, 786)	0.015483241225769256
  (0, 796)	0.08098518487138881
  (0, 829)	0.024012779374943182
  :	:
  (961, 5265)	0.040106642718727006
  (961, 5313)	0.034685407248807254
  (961, 5323)	0.

# <b>#Model Creation</b>

### <b>#Splitting into Train and Test using (Vectorized text & Encoded category)</b>

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(requiredText, df1['Encoded_Category'], test_size=0.35, random_state=42)
print(X_train.shape)

print(X_test.shape)

(625, 6562)
(337, 6562)


### <b>#Training the baseline model and printing its Accuracy</b>
- KNeighbors Classifier
- Multinomial NB

In [32]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

mnb = MultinomialNB()
mnb.fit(X_train,y_train)
y_pred1 = mnb.predict(X_test)
print('Accuracy: ', accuracy_score(y_test,y_pred1)*100, ' % \n')
print(classification_report(y_test, y_pred1))

Accuracy:  90.50445103857567  % 

              precision    recall  f1-score   support

           0       1.00      0.43      0.60         7
           1       1.00      1.00      1.00        12
           2       1.00      0.71      0.83         7
           3       1.00      1.00      1.00        13
           4       1.00      1.00      1.00         9
           5       1.00      0.13      0.24        15
           6       1.00      1.00      1.00        13
           7       1.00      1.00      1.00        10
           8       1.00      0.90      0.95        21
           9       1.00      0.27      0.43        11
          10       1.00      1.00      1.00        10
          11       1.00      1.00      1.00        11
          12       1.00      1.00      1.00        19
          13       1.00      1.00      1.00        10
          14       1.00      1.00      1.00        12
          15       0.67      1.00      0.81        31
          16       1.00      1.00      1.00    

In [33]:
 knc = OneVsRestClassifier(KNeighborsClassifier())
 knc.fit(X_train,y_train)
 y_pred2 = knc.predict(X_test)
 print('Accuracy: ', accuracy_score(y_test,y_pred2)*100, ' %')
 print(classification_report(y_test, y_pred2))

Accuracy:  98.81305637982196  %
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00         7
           3       1.00      1.00      1.00        13
           4       1.00      1.00      1.00         9
           5       1.00      1.00      1.00        15
           6       1.00      0.85      0.92        13
           7       1.00      1.00      1.00        10
           8       1.00      0.90      0.95        21
           9       1.00      1.00      1.00        11
          10       1.00      1.00      1.00        10
          11       1.00      1.00      1.00        11
          12       1.00      1.00      1.00        19
          13       1.00      1.00      1.00        10
          14       1.00      1.00      1.00        12
          15       1.00      1.00      1.00        31
          16       1.00      1.00      1.00      

### <b>#Training the advanced model and printing its Accuracy</b>
- RNN

In [35]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer

# Define the maximum number of words to consider
max_words = 100

# Tokenize and pad sequences
tokenizer = Tokenizer(num_words=max_words, split=' ')
tokenizer.fit_on_texts(df1['Processed_Resume'])
X = tokenizer.texts_to_sequences(df1['Processed_Resume'])
X = pad_sequences(X, maxlen=100)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, df1['Encoded_Category'], test_size=0.25, random_state=42)

# Build the RNN model
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=100))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(label.classes_), activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=150, batch_size=64)

# Evaluate the RNN model
# y_pred3 = model.predict(X_test)
y_pred_rnn_prob = model.predict(X_test)
y_pred3 = y_pred_rnn_prob.argmax(axis=-1)
accuracy_rnn = accuracy_score(y_test, y_pred3)
print('RNN Model Accuracy: {:.2f}%'.format(accuracy_rnn * 100))
print(classification_report(y_test, y_pred3))
# Accuracy is approx 61%


Epoch 1/150


c:\Users\asrit\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.1415 - loss: 3.2004 - val_accuracy: 0.1784 - val_loss: 3.1601
Epoch 2/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.1165 - loss: 3.0724 - val_accuracy: 0.1992 - val_loss: 2.9080
Epoch 3/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.2691 - loss: 2.8078 - val_accuracy: 0.3900 - val_loss: 2.6619
Epoch 4/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.3523 - loss: 2.5327 - val_accuracy: 0.4149 - val_loss: 2.3810
Epoch 5/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.3592 - loss: 2.2866 - val_accuracy: 0.3817 - val_loss: 2.1941
Epoch 6/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.4619 - loss: 2.0971 - val_accuracy: 0.4149 - val_loss: 1.9661
Epoch 7/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.4799 - loss: 1.8771 - val_accuracy: 0.5062 - val_loss: 1.7902
Epoch 8/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.5201 - loss: 1.6875 - val_accuracy: 0.4979 - va

# <b>Prediction & Recommendation System</b>

#### <b>Saving the created models</b>

In [36]:
import pickle
pickle.dump(tfidf,open('tfidf.pkl','wb'))
pickle.dump(mnb, open('mnb.pkl', 'wb'))
pickle.dump(tokenizer, open('rnn_tokenizer.pkl','wb'))
pickle.dump(model, open('rnn.pkl', 'wb'))


In [37]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [38]:
resume_1 = """I am a data scientist specializing in machine
learning, deep learning, and computer vision. With
a strong background in mathematics, statistics,
and programming, I am passionate about
uncovering hidden patterns and insights in data.
I have extensive experience in developing
predictive models, implementing deep learning
algorithms, and designing computer vision
systems. My technical skills include proficiency in
Python, Sklearn, TensorFlow, and PyTorch.
What sets me apart is my ability to effectively
communicate complex concepts to diverse
audiences. I excel in translating technical insights
into actionable recommendations that drive
informed decision-making.
If you're looking for a dedicated and versatile data
scientist to collaborate on impactful projects, I am
eager to contribute my expertise. Let's harness the
power of data together to unlock new possibilities
and shape a better future.
Contact & Sources
Email: 611noorsaeed@gmail.com
Phone: 03442826192
Github: https://github.com/611noorsaeed
Linkdin: https://www.linkedin.com/in/noor-saeed654a23263/
Blogs: https://medium.com/@611noorsaeed
Youtube: Artificial Intelligence
ABOUT ME
WORK EXPERIENCE
SKILLES
NOOR SAEED
LANGUAGES
English
Urdu
Hindi
I am a versatile data scientist with expertise in a wide
range of projects, including machine learning,
recommendation systems, deep learning, and computer
vision. Throughout my career, I have successfully
developed and deployed various machine learning models
to solve complex problems and drive data-driven
decision-making
Machine Learnine
Deep Learning
Computer Vision
Recommendation Systems
Data Visualization
Programming Languages (Python, SQL)
Data Preprocessing and Feature Engineering
Model Evaluation and Deployment
Statistical Analysis
Communication and Collaboration
"""

In [39]:
resume_1 = """
John A. Smith
(555) 555-5555
john.smith@example.com
1 Main Street, Seattle, WA 98133

Profile
A Senior Software Engineer with six years of professional experience, specializing in fullstack development, MySQL, Oracle, and Python. A proven track record of managing large scale software engineering projects to support cloud deployments and integrations.

Professional Experience
Senior Software Engineer, Microsoft, Los Angeles, CA
August 2019-Current

Manage a software engineering team of 15+ personnel to build innovative web applications using Agile-Waterfall methodologies, oversee all aspects of full-stack development, and identify opportunities to enhance the user experience
Identify creative solutions and workflow optimizations to improve deployment timelines and reduce project roadblocks during development lifecycles
Serve as the Microsoft Azure SME for the software engineering department and resolve escalated software issues from junior team members
Software Engineer, Uber, Los Angeles, CA
June 2017-August 2019

Coordinated with a team of 30+ software engineers to re-engineer the system into a PHP-based 3-tier application for a global rideshare company with 300M users
Supported projects to improve geo transit data application tools for drivers and users, which contributed to a 20% increase in user satisfaction
Certifications
MCPS: Microsoft Certified Professional
LPIC-3 Senior Level Linux Certification
Code Camp Trainer
Oracle Certified Professional – Java SE Programmer
Microsoft Certified Solutions Developer
Google Certified Professional Cloud Architect
Key Skills
Application Development
Java/Python/C++/Ruby/Perl/PHP/React/Angular
Full-stack developer
MySQL/Oracle/RedHat/AIX
Analysis and visualization of data structures
Education
Master of Business Administration, Information Systems
California State University, Long Beach, CA, September 2018 – July 2020

Bachelor of Computer Science, Software Engineering Major
University of California, Los Angeles, CA, August 2015 – July 2018, 4.0 GPA


"""

#### <b> Predicting Resume-1 by KNearestClassifier model</b>

In [40]:
"""
import pickle

# Load the trained KNearest classifier model
clf = pickle.load(open('knc.pkl', 'rb'))

# Clean the input resume
cleaned_resume = resumeKeywords(resume_1)

# Transform the cleaned resume using the trained TfidfVectorizer
input_features = tfidf.transform([cleaned_resume])

# Make the prediction using the loaded classifier
prediction_id = clf.predict(input_features)[0]

category_name = category_mapping.get(prediction_id, "Unknown")

print("Predicted Category:", category_name)
print(prediction_id)
"""


'\nimport pickle\n\n# Load the trained KNearest classifier model\nclf = pickle.load(open(\'knc.pkl\', \'rb\'))\n\n# Clean the input resume\ncleaned_resume = resumeKeywords(resume_1)\n\n# Transform the cleaned resume using the trained TfidfVectorizer\ninput_features = tfidf.transform([cleaned_resume])\n\n# Make the prediction using the loaded classifier\nprediction_id = clf.predict(input_features)[0]\n\ncategory_name = category_mapping.get(prediction_id, "Unknown")\n\nprint("Predicted Category:", category_name)\nprint(prediction_id)\n'

#### <b> Predicting Resume-1 by Multinomial Naive Bayes model</b>

In [41]:
import pickle

# Load the trained  Multinomial Naive Bayes model
mnb = pickle.load(open('mnb.pkl', 'rb'))

# Clean the input resume
cleaned_resume = resumeKeywords(resume_1)

# Transform the cleaned resume using the trained TfidfVectorizer
input_features = tfidf.transform([cleaned_resume])

# Make the prediction using the loaded classifier
prediction_id = mnb.predict(input_features)[0]

category_name = category_mapping.get(prediction_id, "Unknown")

print("Predicted Category:", category_name)
print(prediction_id)

Predicted Category: Java Developer
15


#### <b> Predicting Resume-1 by RNN model</b>

In [42]:
import pickle
rnn = pickle.load(open('rnn.pkl', 'rb'))
rnn_tokenizer = pickle.load(open('rnn_tokenizer.pkl', 'rb'))
# Define the maximum number of words to consider
max_words = 5000
cleaned_resume = resumeKeywords(resume_1)  # Clean the input resume

# Tokenize the cleaned input resume
input_sequence = rnn_tokenizer.texts_to_sequences([cleaned_resume])
# Padding the sequences
input_sequence = pad_sequences(input_sequence, maxlen=100)

# Make the prediction using the loaded RNN model
predicted_probabilities = rnn.predict(input_sequence)

prediction_id = predicted_probabilities.argmax(axis=-1)
# Map the category ID to the category name using category_mapping
category_name = category_mapping.get(prediction_id[0], "Unknown")

print("Predicted Category:", category_name)
print("Predicted Category ID:", prediction_id[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step
Predicted Category: Java Developer
Predicted Category ID: 15


In [43]:
job_description_1="""
About the job
Join a top employer and advance your career. Aplin has partnered with an Edmonton-based company to hire a Data Scientist. 

In this exciting role, you will serve as the catalyst for data-driven decision-making, guiding our client toward unparalleled success in lead conversion, customer loyalty, predictive innovation, and inventory optimization. 

Responsibilities:
Dive deep into datasets to unveil the untapped potential in enhancing lead conversion rates.
Develop cutting-edge algorithms that predict customer preferences.
Pioneer loyalty programs that exert a magnetic influence, guaranteeing a continuous stream of returning customers.
Optimize inventory with precision, ensuring the right products are available at the right time.
Qualifications:
Bachelor's degree in Computer Science, Statistics, Applied Math, or related fields; a Master's or PhD will give your credentials an extra boost.
Proficiency in SQL, Python, R, or any other data manipulation language.
Hands-on experience in machine learning, predictive analytics, and various statistical modeling techniques.
Exceptional attention to detail, capable of identifying outliers in datasets with precision.
A passion for storytelling through data, recognizing the importance of context in making impactful decisions.
Outstanding collaboration skills to navigate seamlessly within teams, ensuring smooth progress on your journey.

"""

In [44]:
job_description_1="""
My technical skills include proficiency in
Python, Sklearn, TensorFlow, and PyTorch.
What sets me apart is my ability to effectively
communicate complex concepts to diverse
audiences. I excel in translating technical insights
into actionable recommendations that drive
informed decision-making.

ABOUT ME
WORK EXPERIENCE
SKILLES
NOOR SAEED
LANGUAGES
English
Urdu
Hindi
I am a versatile data scientist with expertise in a wide
range of projects, including machine learning,
recommendation systems, deep learning, and computer
vision. Throughout my career, I have successfully
developed and deployed various machine learning models
to solve complex problems and drive data-driven
decision-making
Machine Learnine
Deep Learning
Computer Vision
Recommendation Systems
Data Visualization
Programming Languages (Python, SQL)
Data Preprocessing and Feature Engineering
Model Evaluation and Deployment
Statistical Analysis
Communication and Collaboration
"""

In [45]:
job_description_1 = """
LMI is seeking a Software Developer or computer science graduate with 7+ years of proven experience in computer vision who has the desire and skill set to design machine vision sensors. You will work in a multi-disciplinary, multi-platform, engineering team (software, electrical, mechanical/optical) that develops new sensor products and supporting infrastructure (manufacturing and test equipment). The ideal candidate will have a passion for leading-edge technology, extensive experience developing production-ready software, strong critical thinking and problem-solving skills, and can work well autonomously yet still communicate effectively with a close-knit group of about 10 engineers. Previous leadership or project management experience is an asset as there is an opportunity to lead a team in this role.

This Senior Software Developer will work in the R&D team and report to the Software Engineering Manager.

Design and develop 3D acquisition algorithms for our sensors to produce 3D data from images
Develop components of our calibration and acquisition pipeline
Characterize and validate prototype sensor performance and integrate final designs with customers
Investigate solutions for challenging acquisition problems. Investigate improvements to our algorithms to enhance the performance of our sensors
Design and develop manufacturing software tools required to build the sensors and control key component performance (e.g. software tasks for focusing and aligning cameras/lasers/projectors, quantifying and adjusting sensor sensitivity, etc.)
Lead technical investigations and produce reports and documentation for senior management
Demonstrate leadership and ownership. Drive projects to completion, participate in frequent peer design and code reviews, and use your expertise to oversee and mentor others in the team
Proactively contribute to and implement continuous improvement initiatives
What do you need to be successful?

Degree / Diploma in Computer Science, Electrical/Computer Engineering or equivalent
5+ years work experience in a disciplined software development environment producing deliverable code
Solid knowledge of C/C++ and C# programming using Microsoft Visual Studio
Expertise in 3D metrology or computer vision (object detection, image restoration, scene reconstruction, signal processing, etc., but excluding machine learning) is required
Experience independently planning and completing complex projects/deliverables in a reliable time frame
Proficient with commonly used scripting languages like Python
Excellent understanding of object-oriented programming
Excellent understanding of commonly used data structures and algorithms (lists, trees, sorting, binning, etc.)
Excellent understanding of math and statistics
Excellent written and verbal communication
Solid understanding of memory management, threading/synchronization, networking
Previous scrum master experience or experience overseeing a small team is an asset
Experience developing for a manufacturing automation environment is an asset
Salary Range: $106,000 - $132,500 - $151,000

Expected Salary: Our typical hiring range will be +/- 10% of the midpoint listed above. Factors influencing this decision include qualifications and market conditions for the role.
"""

# Keywords extraction from provided Job Description and Resume

In [46]:
from sklearn.metrics.pairwise import cosine_similarity
cleaned_resume = resumeKeywords(resume_1)
cleaned_job_description = resumeKeywords(job_description_1)
vectors_1  = tfidf.transform([cleaned_resume])
vectors_2 = tfidf.transform([cleaned_job_description])
similarity_score = cosine_similarity(vectors_1,vectors_2)
print("Similarity Score:", similarity_score)


Similarity Score: [[0.1385646]]


# <b>Bulding Job Category Recommendation System for this dataset</b>

- Shape of the vectors

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_1 = TfidfVectorizer(stop_words='english',max_features=500)

tfidf_1.fit(df1['Processed_Resume'][0:5000])
requiredText  = tfidf_1.transform(df1['Processed_Resume'][0:5000])
print(requiredText)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 75037 stored elements and shape (962, 500)>
  Coords	Values
  (0, 14)	0.11917052199112144
  (0, 19)	0.04683132340033452
  (0, 26)	0.25672550727384724
  (0, 28)	0.28909159624200015
  (0, 30)	0.03871091112683252
  (0, 32)	0.0483605469610782
  (0, 41)	0.05070867379121967
  (0, 49)	0.028590141343798472
  (0, 52)	0.044340118866356124
  (0, 57)	0.05028821612874323
  (0, 61)	0.07654866197897492
  (0, 65)	0.0719702584573859
  (0, 72)	0.05039720138168771
  (0, 73)	0.05070867379121967
  (0, 74)	0.05229137607332689
  (0, 81)	0.013663795865860006
  (0, 86)	0.027070376970601738
  (0, 98)	0.035912642077968696
  (0, 100)	0.04545615080886045
  (0, 103)	0.12534629571459246
  (0, 108)	0.08516543454198569
  (0, 110)	0.08295077192276187
  (0, 112)	0.1472979313066486
  (0, 113)	0.2753802431379583
  (0, 114)	0.026014592123072208
  :	:
  (961, 358)	0.1201836986931062
  (961, 364)	0.0327113401255411
  (961, 370)	0.040841179529759235
  (961, 379)	0.

In [48]:
vectors = requiredText.toarray()
print(vectors.shape)
print(vectors[0])

(962, 500)
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.11917052 0.         0.         0.
 0.         0.04683132 0.         0.         0.         0.
 0.         0.         0.25672551 0.         0.2890916  0.
 0.03871091 0.         0.04836055 0.         0.         0.
 0.         0.         0.         0.         0.         0.05070867
 0.         0.         0.         0.         0.         0.
 0.         0.02859014 0.         0.         0.04434012 0.
 0.         0.         0.         0.05028822 0.         0.
 0.         0.07654866 0.         0.         0.         0.07197026
 0.         0.         0.         0.         0.         0.
 0.0503972  0.05070867 0.05229138 0.         0.         0.
 0.         0.         0.         0.0136638  0.         0.
 0.         0.         0.02707038 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.0359

In [49]:
vectors

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.02412015],
       [0.        , 0.11056736, 0.        , ..., 0.        , 0.        ,
        0.40667888],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.41077046],
       ...,
       [0.02572716, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.09660559],
       [0.16587966, 0.        , 0.        , ..., 0.        , 0.24750229,
        0.        ]], shape=(962, 500))

- Gives the total number of frequent words in the corpus

In [50]:
len(tfidf_1.get_feature_names_out())

500

- Gives the list of most frequent words to less frequent words in the corpus

In [51]:
print(tfidf_1.get_feature_names_out())

['10' '12' '15' '20' '2008' '2010' '2011' '2012' '2013' '2014' '2015'
 '2016' '2017' '2018' '24' '48' 'ability' 'access' 'account' 'action'
 'activity' 'admin' 'administration' 'administrator' 'agile' 'ajax'
 'analysis' 'analyst' 'analytics' 'analyze' 'analyzing' 'android'
 'angular' 'api' 'application' 'architecture' 'area' 'art' 'asp'
 'attending' 'audit' 'automated' 'automation' 'aws' 'bachelor' 'backup'
 'bank' 'banking' 'base' 'based' 'basic' 'basis' 'best' 'billing'
 'blockchain' 'bo' 'board' 'bootstrap' 'branch' 'budget' 'bug' 'build'
 'building' 'business' 'card' 'case' 'center' 'certificate' 'change'
 'check' 'cisco' 'civil' 'client' 'cloud' 'cluster' 'code' 'college' 'com'
 'commerce' 'commercial' 'communication' 'company' 'complete' 'completed'
 'compliance' 'component' 'computer' 'conducted' 'configuration'
 'construction' 'consultancy' 'consultant' 'contract' 'contribution'
 'control' 'controller' 'coordinate' 'coordinating' 'core' 'corporate'
 'cost' 'course' 'create' 'cr

- Looking for most frequent word at certain indexes 

In [52]:
print(tfidf_1.get_feature_names_out()[40])
print(tfidf_1.get_feature_names_out()[20])
print(tfidf_1.get_feature_names_out()[11])
print(tfidf_1.get_feature_names_out()[22])
print(tfidf_1.get_feature_names_out()[33])
print(tfidf_1.get_feature_names_out()[44])

audit
activity
2016
administration
api
bachelor


### Similarity Score (Cosine similarity method)
- For recommending the similar job category we need to identify the nearest vectors of a particular job category vector.
- In order to find out the nearest vectors, we use similarity score and it has two methods (1- Euclidean distance method) & (2- Cosine similarity method)
- Here, I will be using cosine similarity method rather than euclidean distance method because euclidean distance does not perform well on higher dimensions
- The similarity is inverse of distance. [0-1]
- If 1 then similarity is high and if 0 then similarity is low

In [53]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
# This will create result with values as distance/similarity between each vectors
print(similarity)

[[1.         0.21285366 0.29135787 ... 0.04283016 0.06877198 0.09435305]
 [0.21285366 1.         0.50844088 ... 0.01732533 0.18809417 0.10409007]
 [0.29135787 0.50844088 1.         ... 0.03952149 0.28214179 0.19174322]
 ...
 [0.04283016 0.01732533 0.03952149 ... 1.         0.12282746 0.24562413]
 [0.06877198 0.18809417 0.28214179 ... 0.12282746 1.         0.18933751]
 [0.09435305 0.10409007 0.19174322 ... 0.24562413 0.18933751 1.        ]]


In [54]:
print(similarity.shape)
# It finds out distance of each job with every job, and that is the reason why the shape of the similarity shape is (5000,5000)
print(similarity[0])
# Gives similarity score of first job with every job, it can also be seen that the first value is 1 which indicates that the value is the
# similarity score of first job with itself.
print(similarity[1])

(962, 962)
[1.         0.21285366 0.29135787 0.31853294 0.2448193  0.22845056
 0.39632279 0.37209677 0.32211525 0.43922873 1.         0.21285366
 0.29135787 0.31853294 0.2448193  0.22845056 0.39632279 0.37209677
 0.32211525 0.43922873 1.         0.21285366 0.29135787 0.31853294
 0.2448193  0.22845056 0.39632279 0.37209677 0.32211525 0.43922873
 1.         0.21285366 0.29135787 0.31853294 0.2448193  0.22845056
 0.39632279 0.37209677 0.32211525 0.43922873 0.01816578 0.10834576
 0.03095514 0.01709607 0.01709607 0.02570113 0.01125132 0.0463916
 0.00808838 0.04966654 0.12702822 0.01816578 0.10834576 0.03095514
 0.01709607 0.01709607 0.02570113 0.01125132 0.0463916  0.00808838
 0.04966654 0.12702822 0.01816578 0.10834576 0.03095514 0.01709607
 0.01709607 0.02570113 0.01125132 0.0463916  0.00808838 0.04966654
 0.12702822 0.01816578 0.10834576 0.03095514 0.01709607 0.01709607
 0.02570113 0.01125132 0.0463916  0.00808838 0.04966654 0.12702822
 0.02961909 0.05244453 0.05619823 0.09642719 0.11514

In [55]:
df2 = df1
df2 = df2[['Category','Resume','Processed_Resume','Encoded_Category']][0:5000]
df2.head()

,Category,Resume,Processed_Resume,Encoded_Category
0,Data Science,Skills * Programming Languages: Python (pandas...,skill programming language python panda numpy ...,6
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...,education detail may 2013 may 2017 b e uit rgp...,6
2,Data Science,"Areas of Interest Deep Learning, Control Syste...",area interest deep learning control system des...,6
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...,skill r python sap hana tableau sap hana sql s...,6
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab...",education detail mca ymcaust faridabad haryana...,6


In [56]:
df2['Encoded_Category'].value_counts()

Encoded_Category
15    84
23    70
8     55
20    48
24    45
12    44
13    42
6     40
16    40
22    40
18    40
10    40
3     40
1     36
7     33
14    30
11    30
19    30
4     28
9     28
2     26
17    25
5     24
21    24
0     20
Name: count, dtype: int64

### Creating function which will recommend top 5 job categories based on the predicted job category 

1) Initially, if the predicted job category id is given, I will need to find out the index of that job category in my dataset
2) Based on index value, I will be able to know the list of similarity scores of that particular job category with every job category
3) Using Enumerate function to align index and values together while sorting the similarity score in descending order to get top-5 job category

1) Based on given predicted job category id getting the index of that job category in my dataset 

In [63]:
for cat in [0, 38, 26]:
    rows = df2[df2['Encoded_Category'] == cat]
    
    if len(rows) > 0:
        print(cat, rows.index[0])
    else:
        print(cat, "not found")

0 84
38 not found
26 not found


2) Based on the fetched index getting the similarity score list of that job

3) Now as I need the top 5 similar job categories I will need to sort the data in descending order to get top-5.
- However, there is a problem if I am sorting, the index position is getting lost. So, to solve this problem I am using enumerate function (which prints index with value)
- Applying lambda function in that to sort in descending order according to the similarity score and not based on the index value.


In [58]:
# 2.------------------------------- 
print(similarity[0])
print(similarity[38]) 
print(similarity[26]) 
# 3.------------------------------- 
print("-------------------------------------------------------------------------------------------------------------------------------------")
print(list(enumerate(similarity[0]))) 
print(list(enumerate(similarity[38]))) 
print(list(enumerate(similarity[26]))) 
print("-------------------------------------------------------------------------------------------------------------------------------------")
print(sorted(list(enumerate(similarity[0])),reverse = True, key=lambda x:x[1])) 
print(sorted(list(enumerate(similarity[38])),reverse = True, key=lambda x:x[1])) 
print(sorted(list(enumerate(similarity[26])),reverse = True, key=lambda x:x[1])) 

[1.         0.21285366 0.29135787 0.31853294 0.2448193  0.22845056
 0.39632279 0.37209677 0.32211525 0.43922873 1.         0.21285366
 0.29135787 0.31853294 0.2448193  0.22845056 0.39632279 0.37209677
 0.32211525 0.43922873 1.         0.21285366 0.29135787 0.31853294
 0.2448193  0.22845056 0.39632279 0.37209677 0.32211525 0.43922873
 1.         0.21285366 0.29135787 0.31853294 0.2448193  0.22845056
 0.39632279 0.37209677 0.32211525 0.43922873 0.01816578 0.10834576
 0.03095514 0.01709607 0.01709607 0.02570113 0.01125132 0.0463916
 0.00808838 0.04966654 0.12702822 0.01816578 0.10834576 0.03095514
 0.01709607 0.01709607 0.02570113 0.01125132 0.0463916  0.00808838
 0.04966654 0.12702822 0.01816578 0.10834576 0.03095514 0.01709607
 0.01709607 0.02570113 0.01125132 0.0463916  0.00808838 0.04966654
 0.12702822 0.01816578 0.10834576 0.03095514 0.01709607 0.01709607
 0.02570113 0.01125132 0.0463916  0.00808838 0.04966654 0.12702822
 0.02961909 0.05244453 0.05619823 0.09642719 0.11514494 0.00679

- Final Custom function

In [59]:
def recommend(category):
    #Based on given predicted job category id getting the index of that job category in my dataset 
    index = df2[df2['Encoded_Category'] == category].index[0] 

    #Based on the fetched index getting the similarity score list of that job
    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1]) 
    # print(distances) # For testing purpose
    
    unique_set = set([])
    similarity_score = []
    
    for i in distances[0:100]: 
        if len(unique_set) < 5:
            job = df2.iloc[i[0]].Category
            score = "{:.2f}".format(i[1])
            if job not in unique_set and score != "1.00":
                unique_set.add(job)
                similarity_score.append(i[1])
                print(f"{job}, {score}") # Print each unique value and its corresponding similarity score
        else:
            break

In [60]:
print(category_mapping.get(0))
recommend(0)

Advocate
Civil Engineer, 0.42
Advocate, 0.33
Health and fitness, 0.28
Business Analyst, 0.19
Database, 0.17


In [65]:
print(category_mapping.get(15))
recommend(15)

Java Developer
Java Developer, 0.55
HR, 0.36
Automation Testing, 0.27
Blockchain, 0.23
DotNet Developer, 0.21


In [66]:
print(category_mapping.get(18))
recommend(18)

Operations Manager
Operations Manager, 0.53
PMO, 0.52
DevOps Engineer, 0.48
Business Analyst, 0.40
HR, 0.34
